# Stress Prediction (Improved - Better Generalization)
Focus: reduce overfitting + improve private LB

In [4]:
# Install (if needed)
# !pip install lightgbm scikit-learn pandas numpy

In [5]:
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupKFold
from sklearn.metrics import balanced_accuracy_score

import lightgbm as lgb

In [6]:
train_data = pd.read_csv('train-sensor.csv')
train_label = pd.read_csv('train-label.csv')

test_data = pd.read_csv('test-sensor.csv')
test_label = pd.read_csv('test-label.csv')

print(train_data.shape, train_label.shape)

(4694400, 8) (815, 4)


In [7]:
def extract_features(data, labels, window=15000):
    features = []

    for _, row in labels.iterrows():
        pid = row['pid']
        ts = row['timestamp']

        subset = data[
            (data['pid'] == pid) &
            (data['timestamp'] >= ts - window) &
            (data['timestamp'] <= ts)
        ]

        if len(subset) < 5:
            continue

        feat = {'id': row['id'], 'pid': pid}

        for col in ['accel_x','accel_y','accel_z','eda','heart_rate','temperature']:
            vals = subset[col]

            feat[f'{col}_mean'] = vals.mean()
            feat[f'{col}_std'] = vals.std()
            feat[f'{col}_min'] = vals.min()
            feat[f'{col}_max'] = vals.max()
            feat[f'{col}_range'] = vals.max() - vals.min()
            feat[f'{col}_median'] = vals.median()

        # HRV proxy (important for stress)
        hr = subset['heart_rate'].values
        feat['hrv_std'] = np.std(np.diff(hr)) if len(hr) > 1 else 0

        # EDA change intensity
        eda = subset['eda'].values
        feat['eda_diff_mean'] = np.mean(np.abs(np.diff(eda))) if len(eda) > 1 else 0

        if 'stress' in row:
            feat['stress'] = row['stress']

        features.append(feat)

    return pd.DataFrame(features)

In [8]:
train_feat = extract_features(train_data, train_label)

X = train_feat.drop(columns=['id','pid','stress'])
y = train_feat['stress']
groups = train_feat['pid']

X = X.fillna(0)

print("Train shape:", X.shape)

Train shape: (815, 38)


In [9]:
gkf = GroupKFold(n_splits=5)

scores = []
models = []

for fold, (train_idx, val_idx) in enumerate(gkf.split(X, y, groups)):
    print(f"Fold {fold+1}")

    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = lgb.LGBMClassifier(
        n_estimators=500,
        learning_rate=0.03,
        max_depth=5,
        num_leaves=25,
        subsample=0.8,
        colsample_bytree=0.8,
        class_weight='balanced',
        random_state=42
    )

    model.fit(X_train, y_train)

    pred = model.predict(X_val)
    score = balanced_accuracy_score(y_val, pred)
    print("BA:", score)

    scores.append(score)
    models.append(model)

print("CV Balanced Accuracy:", np.mean(scores))

Fold 1
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000248 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 6905
[LightGBM] [Info] Number of data points in the train set: 663, number of used features: 38
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning

/Users/dayana/git repo/machine-learning-class/lab3/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:2924: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

/Users/dayana/git repo/machine-learning-class/lab3/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:2924: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


BA: 0.44029850746268656
Fold 4
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000193 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 6389
[LightGBM] [Info] Number of data points in the train set: 616, number of used features: 38
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: 

/Users/dayana/git repo/machine-learning-class/lab3/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:2924: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

In [10]:
test_feat = extract_features(test_data, test_label)

X_test = test_feat.drop(columns=['id','pid']).fillna(0)

# Align columns
X_test = X_test.reindex(columns=X.columns, fill_value=0)

print("Test shape:", X_test.shape)

Test shape: (1028, 38)


In [11]:
# Ensemble prediction (average)
preds = []

for model in models:
    preds.append(model.predict_proba(X_test))

avg_pred = np.mean(preds, axis=0)
test_pred = np.argmax(avg_pred, axis=1)

In [12]:
submission = pd.DataFrame({
    'id': test_feat['id'],
    'stress': test_pred
})

submission.to_csv('submissiong.csv', index=False)
print("submission.csv saved!")

submission.csv saved!
